# Imagination-Augmented Agent (I2A)

An **Imagination-Augmented Agent (I2A)** is a type of reinforcement learning agent that enhances traditional policy-based methods by incorporating a mechanism for imagining future trajectories. Unlike agents that rely solely on direct interaction with the environment to learn, an I2A agent uses an internal "environment model" to simulate potential future outcomes of different actions.

Here's a breakdown of the key components:

*   **Environment Model:** This model learns to predict the next state and reward given a current state and action. It allows the agent to "imagine" what would happen if it took a particular action without actually performing that action in the real environment.
*   **Rollout Encoder:** This component processes the imagined trajectories generated by the environment model. It compresses the sequence of imagined states into a compact representation.
*   **Policy Network (Actor-Critic):** This network takes the current state *and* the encoded imagined trajectories as input. It has two outputs:
    *   **Actor:** Determines the probability of taking each possible action.
    *   **Critic:** Estimates the value of the current state.

By combining real observations with imagined future outcomes, the I2A agent can make more informed decisions, especially in environments where exploration is difficult or costly. The imagination process helps the agent to look ahead and evaluate the potential consequences of its actions before committing to them.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import random
import numpy as np

Let's use the same simple environment used for Dyna-Q.

The agent has 4 possible actions:
- Move Up
- Move Right
- Move Down
- Move Left

The task is to reach a certain goal from a start position.

The reward is 1 if the agent reaches the goal and 0 otherwise.

Note: we're not using Gymansium's Agent API, thus the environment does not completely follow the usual conventions!

In [ ]:
# Grid World Environment (Unchanged)
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.reset()

    def reset(self):
        self.agent_pos = [random.randint(0, self.size-1), random.randint(0, self.size-1)]
        self.goal_pos = [self.size-1, self.size-1]
        # Ensure agent does not start at the goal
        while self.agent_pos == self.goal_pos:
             self.agent_pos = [random.randint(0, self.size-1), random.randint(0, self.size-1)]
        return self._get_state()

    def _get_state(self):
        # The state now includes agent and goal positions
        state = np.zeros((self.size, self.size), dtype=np.float32)
        state[self.agent_pos[0], self.agent_pos[1]] = 1.0
        state[self.goal_pos[0], self.goal_pos[1]] = 0.5
        return torch.tensor(state).flatten().unsqueeze(0)


    def step(self, action):
        if action == 0 and self.agent_pos[1] > 0:  # left
            self.agent_pos[1] -= 1
        elif action == 1 and self.agent_pos[1] < self.size - 1:  # right
            self.agent_pos[1] += 1
        elif action == 2 and self.agent_pos[0] > 0:  # up
            self.agent_pos[0] -= 1
        elif action == 3 and self.agent_pos[0] < self.size - 1:  # down
            self.agent_pos[0] += 1

        done = self.agent_pos == self.goal_pos
        reward = 1.0 if done else -0.1 # Small negative reward to encourage efficiency
        return self._get_state(), reward, done

The next cell defines the EnvModel class, which serves as the agent's internal model of the environment. This model is trained to predict the next state and the reward given the current state and the action taken by the agent. This allows the agent to "imagine" the consequences of its actions without actually interacting with the real environment.

In [ ]:
# The model now predicts the next state and reward
class EnvModel(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.action_size = action_size
        self.fc = nn.Sequential(
            nn.Linear(state_size + action_size, 128),
            nn.ReLU(),
            nn.Linear(128, state_size + 1) # Predicts next state AND reward
        )

    def forward(self, state, action):
        # One-hot encode the action
        action_one_hot = torch.zeros(state.size(0), self.action_size)
        action_one_hot[torch.arange(state.size(0)), action.long()] = 1.0

        x = torch.cat([state, action_one_hot], dim=1)
        output = self.fc(x)
        next_state_pred = output[:, :-1]
        reward_pred = output[:, -1].unsqueeze(1)
        return next_state_pred, reward_pred

The RolloutEncoder class processes the imagined trajectories generated by the environment model. It uses a GRU (Gated Recurrent Unit) to handle the sequence of imagined states over time and compresses this sequence into a compact, fixed-size representation. This encoded representation captures the essential information from the imagined future to be used by the policy network.

The output of the RolloutEncoder, which is a compact representation of the imagined trajectories, is concatenated with the agent's current real state. This combined information (real state + encoded imagination) is then fed into the ActorCriticNet. The ActorCriticNet uses this combined input to determine the best action to take (Actor) and to estimate the value of the current state (Critic). This allows the agent to make decisions based on both its current observation and its imagined future outcomes.

In [ ]:
# This is now more distinct and uses a recurrent layer (GRU) to process a sequence
class RolloutEncoder(nn.Module):
    def __init__(self, state_size, hidden_size=64):
        super().__init__()
        # GRU is well-suited for processing sequences of imagined steps
        self.gru = nn.GRU(state_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 32) # Encoded representation

    def forward(self, imagined_trajectories):
        # imagined_trajectories shape: (batch_size, num_rollouts, rollout_depth, state_size)
        batch_size, num_rollouts, rollout_depth, state_size = imagined_trajectories.shape
        # Process each rollout for each action
        reshaped_trajs = imagined_trajectories.view(batch_size * num_rollouts, rollout_depth, state_size)
        _, h_n = self.gru(reshaped_trajs)
        h_n = h_n.squeeze(0) # Get the final hidden state
        encoded = torch.relu(self.fc(h_n))
        # Reshape back to (batch_size, num_rollouts * encoded_size)
        return encoded.view(batch_size, -1)

The ActorCriticNet class is the core policy network of the I2A agent. It's designed to act as both an actor (selecting actions) and a critic (evaluating the state).

Here's a breakdown:

- Input: It takes two inputs:
  1. the current state (the real observation from the environment) and
  2. the encoded_imagination (the compact representation of imagined future trajectories produced by the RolloutEncoder).

 - Base Network (self.base): This is a shared feed-forward network that processes the concatenated real state and encoded imagination. It consists of a linear layer followed by a ReLU activation. This part of the network learns a combined representation that incorporates both the current situation and the potential future outcomes.

- Actor Head (self.actor): This is a linear layer that takes the output of the base network and outputs action_logits. These logits represent the raw scores for each possible action. These scores are typically passed through a softmax function (though done outside this class, usually in the training or action selection logic) to get a probability distribution over the actions. The agent then samples an action from this distribution.

- Critic Head (self.critic): This is another linear layer that takes the output of the base network and outputs a single value, state_value. This value is the network's estimate of the expected cumulative future reward from the current state. This value is used in the training process to calculate the advantage (how much better or worse an action was than expected), which helps in updating the actor.

In [ ]:
# This now contains two heads: one for action selection (Actor) and one for state valuation (Critic)
class ActorCriticNet(nn.Module):
    def __init__(self, state_size, encoded_size, action_size):
        super().__init__()
        # Shared base network processes both real state and imagined encoding
        self.base = nn.Sequential(
            nn.Linear(state_size + encoded_size, 128),
            nn.ReLU()
        )
        # The Actor head outputs probabilities for each action
        self.actor = nn.Linear(128, action_size)
        # The Critic head outputs a single value for the current state
        self.critic = nn.Linear(128, 1)

    def forward(self, state, encoded_imagination):
        x = torch.cat([state, encoded_imagination], dim=1)
        base_out = self.base(x)
        action_logits = self.actor(base_out)
        state_value = self.critic(base_out)
        return action_logits, state_value

The I2AAgent orchestrates the interaction between the environment, the internal environment model, the imagination process, and the policy network to learn how to take actions that lead to higher rewards, leveraging the ability to "look ahead" through imagination.

Here's a breakdown of its key methods:

- **\_\_init\_\_**: Initializes the agent with the environment model, rollout encoder, and policy network. It also sets up separate optimizers for the environment model and the policy/encoder.
- **imagine_trajectories**: This method uses the EnvModel to simulate future states and rewards for a given depth (rollout_depth) for each possible action, without interacting with the real environment.
- **select_action**: This method first calls imagine_trajectories to get the imagined future. Then, it uses the RolloutEncoder to process these imagined trajectories into a compact representation. Finally, it feeds this encoded imagination and the current state into the ActorCriticNet to get action logits and samples an action from the resulting probability distribution.
- **train**: This method performs a training step for both the environment model and the policy network. It updates the EnvModel based on the real environment interaction (state, action, reward, next_state) and updates the ActorCriticNet and RolloutEncoder using the actor-critic learning algorithm, incorporating the imagined trajectories.

The train function uses several loss functions to train the different models included in the actor-critic model and the imagination process:

**Environment Model Loss**:

This loss function is a combination of the Mean Squared Error (MSE) between the predicted next state and the actual next state, and the MSE between the predicted reward and the actual reward:

$L_{model} = \text{MSE}(\hat{s}_{t+1}, s_{t+1}) + \text{MSE}(\hat{r}_t, r_t)$

Where:

  - $\hat{s}_{t+1}$ is the predicted next state by the EnvModel.
  - $s_{t+1}$ is the actual next state from the environment.
  - $\hat{r}_t$ is the predicted reward by the EnvModel.
  - $r_t$ is the actual reward from the environment.

**Actor-Critic Policy Loss**:

This loss function is a combination of the actor loss (policy gradient loss) and the critic loss (MSE loss for the value function).
$L_{policy} = L_{actor} + L_{critic}$
The actor loss is calculated using the policy gradient objective, weighted by the advantage:
$L_{actor} = -\log(\pi(a_t | s_t, i_t)) \cdot A_t$

Where:

- $\pi(a_t | s_t, i_t)$ is the probability of taking action $a_t$ in state $s_t$, conditioned on the encoded imagination $i_t$, as output by the ActorCriticNet.
- $A_t$ is the advantage, which is the difference between the target value and the predicted state value: $A_t = G_t - V(s_t)$.
- $G_t$ is the target value, calculated using the received reward and the predicted value of the next state: $G_t = r_t + \gamma \cdot V(s_{t+1})$$G_t = r_t + \gamma \cdot V(s_{t+1})$.
- $V(s_t)$ is the predicted value of the current state by the ActorCriticNet.
- $\gamma$ is the discount factor.

The critic loss is the MSE between the advantage and zero (or equivalently, the squared advantage):

$L_{critic} = A_t^2$

By minimizing the policy loss, the agent learns to adjust its policy to favor actions that lead to higher-than-expected rewards and to improve its estimation of state values.

In [ ]:
# The I2A Agent
class I2AAgent:
    def __init__(self, state_size, action_size, rollout_depth=3, lr=0.001, gamma=0.99):
        self.state_size = state_size
        self.action_size = action_size
        self.rollout_depth = rollout_depth
        self.gamma = gamma

        self.env_model = EnvModel(state_size, action_size)
        self.encoder = RolloutEncoder(state_size)
        self.policy = ActorCriticNet(state_size, 32 * action_size, action_size)

        # Separate optimizers for the model and the policy
        self.model_optimizer = optim.Adam(self.env_model.parameters(), lr=lr)

        # Next we add the parameters of the RolloutEncoder to the policy optimizer
        # thus we force the optimization of both networks simulatenously!
        self.policy_optimizer = optim.Adam(list(self.policy.parameters()) + list(self.encoder.parameters()), lr=lr)

    def imagine_trajectories(self, state):
        """Generates imagined future trajectories for all possible actions."""
        imagined_trajectories = []
        with torch.no_grad(): # No need to track gradients here
            for a in range(self.action_size):
                rollout = []
                current_state = state
                action = torch.tensor([a])
                for _ in range(self.rollout_depth):
                    # We use the environment model to estimate the next state
                    next_state_pred, _ = self.env_model(current_state, action)
                    rollout.append(next_state_pred)
                    current_state = next_state_pred
                imagined_trajectories.append(torch.cat(rollout, dim=0).unsqueeze(0))
        # Shape: (1, num_actions, rollout_depth, state_size)
        return torch.cat(imagined_trajectories, dim=0).unsqueeze(0)

    def select_action(self, state):
        """Selects an action based on policy and imagination."""
        # generate a trajectory from this state using the environment model
        imagined_encodings = self.encoder(self.imagine_trajectories(state))
        # get the action logits
        action_logits, _ = self.policy(state, imagined_encodings)
        # Use a categorical distribution to allow for exploration
        dist = Categorical(logits=action_logits)
        # and sample the best action probabilistically
        action = dist.sample()
        return action.item()

    def train(self, state, action, reward, next_state, done):
        """Performs a full training step for both the model and the policy."""
        action = torch.tensor([action])
        reward = torch.tensor([reward], dtype=torch.float32)
        done_mask = torch.tensor([1 - done], dtype=torch.float32)

        # 1. Train the Environment Model to match the next state and reward
        next_state_pred, reward_pred = self.env_model(state, action)
        model_loss = nn.functional.mse_loss(next_state_pred, next_state) + \
                     nn.functional.mse_loss(reward_pred, reward)

        self.model_optimizer.zero_grad()
        model_loss.backward()
        self.model_optimizer.step()

        # 2. Train the Actor-Critic Policy
        # Get the policy's evaluation of the current state and action
        imagined_encodings = self.encoder(self.imagine_trajectories(state))
        action_logits, state_value = self.policy(state, imagined_encodings)

        # Get the policy's evaluation of the next state to calculate the target value
        with torch.no_grad():
            next_imagined_encodings = self.encoder(self.imagine_trajectories(next_state))
            _, next_state_value = self.policy(next_state, next_imagined_encodings)
            target_value = reward + self.gamma * next_state_value * done_mask

        # Calculate the advantage (how much better was the action than expected)
        advantage = target_value - state_value

        # Calculate losses
        dist = Categorical(logits=action_logits)
        actor_loss = -(dist.log_prob(action) * advantage.detach()) # Policy gradient loss
        critic_loss = advantage.pow(2) # MSE loss for the critic

        policy_loss = actor_loss + critic_loss

        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        self.policy_optimizer.step()

Next we can perform the training loop as usual:

In [ ]:
# The Training Loop ---
env = GridWorld(size=5)
state_size = env.size * env.size
agent = I2AAgent(state_size=state_size, action_size=4)
total_rewards = []

print("Starting training...")
for episode in range(1500):
    state = env.reset()
    total_reward = 0
    for t in range(50): # Increased episode length
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)

        agent.train(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward
        if done:
            break
    total_rewards.append(total_reward)

    if episode % 100 == 0:
        print(f"Episode {episode}, Avg. Total Reward: {np.mean(total_rewards[-100:])}")

print("Training finished.")

We can test the trained agent

In [ ]:
# Final test
state = env.reset()
trajectory = [env.agent_pos.copy()]
print(f"Goal is at: {env.goal_pos}, Starting at: {env.agent_pos.copy()}")
for t in range(20):
    action = agent.select_action(state)
    state, _, done = env.step(action)
    trajectory.append(env.agent_pos.copy())
    if done:
        print("Goal reached!")
        break
print("Final test trajectory:", trajectory)